# Causal Inference ~ Group 10

## Problem 6 ~ Card and Krueger (1994) Difference-in-Differences

We replicate the Card and Krueger (1994) difference-in-differences analysis of the New Jersey minimum wage increase using the dataset `DinD_ex.dta`. The variables are:

- `fte` (Y): full-time-equivalent number of employees at a restaurant.
- `nj` (G): indicator for being in New Jersey (1) vs. Pennsylvania (0).
- `after` (T): indicator for the post-minimum-wage-change period (1) vs. pre (0).
- `njafter` (D): interaction = NJ * after, i.e. 1 for NJ restaurants observed after the change.
- `sheet`: unique restaurant identifier.

# Part A ~ Replicating the DiD Regression

We estimate the canonical DiD specification with robust (HC1) standard errors:

$$\text{fte}_{i,t} = \beta_0 + \beta_1 \cdot \text{nj}_i + \beta_2 \cdot \text{after}_t + \beta_3 \cdot \text{njafter}_{i,t} + \varepsilon_{i,t}$$

The DiD estimate of the average treatment effect on the treated is the coefficient $\beta_3$ on `njafter`.

In [1]:
import pandas as pd
import statsmodels.api as sm

# Load data
df = pd.read_stata("causal_data/DinD_ex.dta")

# DiD regression with robust standard errors
Y = df["fte"]
X = df[["nj", "after", "njafter"]]
X = sm.add_constant(X)

did_model = sm.OLS(Y, X, missing='drop').fit(cov_type='HC1')
print(did_model.summary())

# Explicit values referenced in the interpretation below
print("\n===== Key values referenced in interpretation =====")
print(f"Coefficient on njafter:  {did_model.params['njafter']:.4f}")
print(f"Robust SE on njafter:    {did_model.bse['njafter']:.4f}")
print(f"z-statistic on njafter:  {did_model.tvalues['njafter']:.4f}")
print(f"p-value on njafter:      {did_model.pvalues['njafter']:.4f}")
print(f"Coefficient on nj:       {did_model.params['nj']:.4f}")
print(f"Coefficient on after:    {did_model.params['after']:.4f}")
print(f"Constant:                {did_model.params['const']:.4f}")

                            OLS Regression Results                            
Dep. Variable:                    fte   R-squared:                       0.009
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     1.315
Date:                Thu, 30 Apr 2026   Prob (F-statistic):              0.268
Time:                        21:17:01   Log-Likelihood:                -2519.3
No. Observations:                 698   AIC:                             5047.
Df Residuals:                     694   BIC:                             5065.
Df Model:                           3                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         20.3000      1.502     13.519      0.0

## Part A Interpretation
The DiD estimate of the New Jersey minimum-wage increase on full-time-equivalent employment is the coefficient on `njafter`: **2.3287 employees**, with a robust standard error of **1.9308**. The estimate is positive -- the opposite sign of what classical labor-demand theory predicts -- and is statistically indistinguishable from zero (z = **1.2061**, p = **0.2278**). This replicates the central Card and Krueger (1994) finding: raising the minimum wage in New Jersey did not reduce fast-food employment relative to Pennsylvania, and if anything the point estimate suggests a small employment increase. The coefficient on `nj` (**-2.9989**) shows NJ restaurants were slightly smaller pre-change, and the coefficient on `after` (**-2.0462**) shows PA restaurants shrank over the period; the DiD nets these two trends out to isolate the NJ-specific change.

## Part B ~ Confirming Equivalence with the Means-Comparison DiD

We confirm that the regression coefficient on `njafter` equals the textbook DiD formula:

$$DID = \big(E[Y_{1,1}] - E[Y_{1,0}]\big) - \big(E[Y_{0,1}] - E[Y_{0,0}]\big)$$

where the first subscript is `nj` and the second is `after`.

In [2]:
# Group means of fte by (nj, after)
means = df.groupby(["nj", "after"])["fte"].mean().unstack()
print("Group means of fte:")
print(means)

y11 = df[(df["nj"] == 1) & (df["after"] == 1)]["fte"].mean()
y10 = df[(df["nj"] == 1) & (df["after"] == 0)]["fte"].mean()
y01 = df[(df["nj"] == 0) & (df["after"] == 1)]["fte"].mean()
y00 = df[(df["nj"] == 0) & (df["after"] == 0)]["fte"].mean()

did_from_means = (y11 - y10) - (y01 - y00)
did_from_reg = did_model.params['njafter']

print(f"\nE[Y_NJ_after]   = {y11:.4f}")
print(f"E[Y_NJ_before]  = {y10:.4f}")
print(f"E[Y_PA_after]   = {y01:.4f}")
print(f"E[Y_PA_before]  = {y00:.4f}")
print(f"\nDiD from means:    ({y11:.4f} - {y10:.4f}) - ({y01:.4f} - {y00:.4f}) = {did_from_means:.4f}")
print(f"njafter coefficient from regression in Part A:           {did_from_reg:.4f}")
print(f"Difference (should be ~0):                               {did_from_means - did_from_reg:.4f}")

Group means of fte:
after        0.0        1.0
nj                         
0.0    20.299999  18.253845
1.0    17.301056  17.583628

E[Y_NJ_after]   = 17.5836
E[Y_NJ_before]  = 17.3011
E[Y_PA_after]   = 18.2538
E[Y_PA_before]  = 20.3000

DiD from means:    (17.5836 - 17.3011) - (18.2538 - 20.3000) = 2.3287
njafter coefficient from regression in Part A:           2.3287
Difference (should be ~0):                               0.0000


## Part B Interpretation
The means-based DiD calculation produces the same estimate as the regression in Part A: **2.3287** in both cases (difference is 0.0000). This confirms the well-known algebraic equivalence -- when the DiD regression includes only the two main effects and their interaction with no other covariates, the coefficient on the interaction term is mechanically identical to the four-cell mean-comparison estimator. The standard errors and inferential machinery differ across the two approaches, but the point estimate of the ATT is the same.

## Part C ~ Checking for Serial Autocorrelation

We pivot the panel to one row per restaurant (`sheet`) and estimate the within-restaurant correlation between pre-period `fte` and post-period `fte`. A high correlation indicates serial autocorrelation in the outcome at the restaurant level, which has implications for inference.

In [3]:
# Reshape so each restaurant has one pre and one post observation side-by-side
wide = df.pivot(index="sheet", columns="after", values="fte")
wide.columns = ["fte_pre", "fte_post"]

print("Sample of wide-format data (one row per restaurant):")
print(wide.head())

n_both = wide.dropna().shape[0]
print(f"\nNumber of restaurants observed in both periods: {n_both}")

cov_pre_post = wide["fte_pre"].cov(wide["fte_post"])
corr_pre_post = wide["fte_pre"].corr(wide["fte_post"])

print(f"\nCov(fte_pre, fte_post)  = {cov_pre_post:.4f}")
print(f"Corr(fte_pre, fte_post) = {corr_pre_post:.4f}")

Sample of wide-format data (one row per restaurant):
       fte_pre  fte_post
sheet                   
1         31.0      40.0
2         13.0      12.5
3         12.5       7.5
4         16.0      20.0
5         20.0      25.0

Number of restaurants observed in both periods: 349

Cov(fte_pre, fte_post)  = 43.8119
Corr(fte_pre, fte_post) = 0.5478


## Part C Interpretation
The within-restaurant correlation between pre-period `fte` and post-period `fte` is **0.5478** across **349** restaurants observed in both periods, with a covariance of **43.8119**. This is clear evidence of serial autocorrelation: a restaurant that was relatively large before the minimum-wage change tends to also be relatively large after it. In other words, the residuals from the DiD regression are not independent across the two observations of the same restaurant -- they share a persistent restaurant-specific component.

**Implication for the standard errors in Part A:** the HC1-robust standard errors used in the regression assume observations are independent (only allowing for heteroskedasticity, not within-unit dependence). With autocorrelation of ~0.55 across the two periods for each restaurant, the robust SE in Part A is too small -- it overstates the effective sample size by treating the two observations of each restaurant as independent draws when they are not. The proper fix is to cluster the standard errors at the restaurant (`sheet`) level, which would inflate the reported SE on `njafter` and likely widen the confidence interval. The point estimate of 2.3287 is unaffected, but inference about whether it differs from zero becomes more conservative.